# CCE PoC v5: full study on RepoBench cross_file_first (3 models)

**Goal.** Run the gating-signal evaluation across all three 7B code LLMs (Qwen2.5-Coder, DeepSeek-Coder, CodeLlama) on a contamination-resistant benchmark (RepoBench Python `cross_file_first` split, n=250 stratified per model). All analysis (AUC + bootstrap CIs + end-to-end gated retrieval curves) runs on the *filtered subset* where retrieval-augmented generation is sufficient to produce a correct answer — the regime where gating signals can plausibly do useful work.

**Why this design.** The v4 probe (Qwen only, n=50 → n=250) hit RED on aggregate metrics because 80% of RepoBench tasks fail both with and without retrieval (the both-fail floor on a 7B model). The verdict turned GREEN once we conditioned on `always_correct=1`. This study runs the same conditional analysis across all three models with the full ablation grid (9 arms) and bootstrap CIs, plus end-to-end gated retrieval curves to answer the *systems* question (not just the classification one).

**What this notebook produces, per model:**
- `phase1_never_retrieve__{slug}.json` — cold answers + per-token features for all 250 tasks
- `phase2_always_retrieve__{slug}.json` — retrieval-augmented answers for all 250
- `phase4_se_samples__{slug}.json` — 5 sampled generations per filtered task
- `phase5_nli__{slug}.json` — DeBERTa-MNLI clustering features per filtered task
- `results__{slug}.json` — final per-model AUC table + bootstrap CIs + gated retrieval curves

**Wall-clock estimate (A100, per model).** ~3 hrs each. ~9 hrs across all three. Resumable: per-phase Drive cache, content-stable keys (no v4 cache-collision bug).

**Bug-prevention checklist applied:**
- Cache keys are `md5(repo + question[:500])`, not sequential IDs that change with TARGET_N.
- `CCEFeatureExtractor` is constructed per task with `(code_ids, lang_ids)` from `build_partition_indices`, not the broken `partition='lenient'` kwarg.
- `generate_with_features` is called with the correct (model, tokenizer, prompt, extractor, max_new_tokens=N) signature — no spurious `probe_layers` positional arg.
- DeBERTa-large-MNLI runs in fp32 with `torch.autocast(enabled=False)` — fp16 silently NaNs on its attention.
- RepoBench loaded with `cross_file_first` split (not `test`).
- Verifier normalizes markdown code fences + collapsed whitespace before substring match — handles chat-model output that wraps code in ``` blocks.
- Multi-model orchestration via subprocess (clean VRAM between runs).

## Phase 0 — Environment setup

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn datasets

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemError('No GPU detected. Switch runtime → GPU (A100 preferred, L4 fallback).')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true
!git log --oneline -5

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
    print('HF login OK (needed for CodeLlama; Qwen and DeepSeek are public).')
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/cce_poc_v5'
!mkdir -p {OUT_DIR}
print(f'will write per-model artifacts to {OUT_DIR}')

## Phase 1 — Run all three models sequentially

Each model is a separate `subprocess` call to `cce_poc_v5.py`. This guarantees clean VRAM between runs and isolates failures (one model crashing doesn't kill the others). Per-phase Drive cache makes each run resumable on disconnect.

**Order rationale.** Qwen first (smallest VRAM peak, fastest, the v4 model where we already have signal). Then DeepSeek (public, stable). CodeLlama last (gated; if HF login is broken we want to know after the others succeeded).

**Re-running this cell is safe** — caches resume.

In [ ]:
import time, subprocess, os

MODELS = [
    'Qwen/Qwen2.5-Coder-7B-Instruct',
    'deepseek-ai/deepseek-coder-7b-instruct-v1.5',
    'codellama/CodeLlama-7b-Instruct-hf',
]

%cd /content/reposynth
for i, model in enumerate(MODELS, 1):
    print(f"\n{'='*70}\n[{i}/{len(MODELS)}] {model}\n{'='*70}", flush=True)
    t0 = time.time()
    rc = subprocess.call([
        'python', '/content/reposynth/research/paper/cce_poc_v5.py',
        '--model', model,
        '--n-tasks', '250',
        '--out-dir', OUT_DIR,
    ])
    dt = (time.time() - t0) / 60
    if rc == 0:
        print(f"\n[{i}/{len(MODELS)}] OK   {model}  ({dt:.1f} min)", flush=True)
    else:
        print(f"\n[{i}/{len(MODELS)}] FAIL {model}  rc={rc}  ({dt:.1f} min)", flush=True)
        print('Continuing with the remaining models. Re-run this cell later to retry.', flush=True)

print('\nAll models attempted. Inspect per-model results below.')

## Phase 2 — Inspect per-model results

Loads each `results__{slug}.json` and prints the headline numbers.

In [ ]:
import json, re
from pathlib import Path

out_dir = Path(OUT_DIR)
results_files = sorted(out_dir.glob('results__*.json'))
if not results_files:
    print('No results files found yet. Run Phase 1 first.')
else:
    for f in results_files:
        with open(f) as fh:
            r = json.load(fh)
        print(f"\n=== {r['model']} ===")
        ov = r['overall']
        fs = r['filtered_subset']
        print(f"  overall n={r['n_total']}: never={ov['never_retrieve_accuracy']:.3f}, "
              f"always={ov['always_retrieve_accuracy']:.3f}, gap={ov['gap_pp']:+.1f}pp")
        print(f"  filtered subset: n={fs['n']}, positives={fs['n_positives']} "
              f"({fs['positive_rate']:.1%}), retention={r['filter_retention_rate']:.1%}")
        print(f"\n  Univariate AUC (95% CI):")
        for key, u in r['univariate_auc'].items():
            print(f"    {u['name']:<22} {u['auc']:.3f} [{u['ci_lo']:.3f}, {u['ci_hi']:.3f}]")
        print(f"\n  Arm AUC (LOO-LR, 95% CI):")
        for arm, a in r['arm_auc'].items():
            print(f"    {arm:<22} {a['auc']:.3f} [{a['ci_lo']:.3f}, {a['ci_hi']:.3f}]  "
                  f"({a['n_features']}f)")

## Phase 3 — Cross-model comparison table

Best signal per model + cross-model AUC table for the headline arms.

In [ ]:
import json
from pathlib import Path

out_dir = Path(OUT_DIR)
results_files = sorted(out_dir.glob('results__*.json'))
all_results = {}
for f in results_files:
    with open(f) as fh:
        r = json.load(fh)
    all_results[r['model']] = r

if not all_results:
    print('No results to aggregate.')
else:
    HEADLINE_ARMS = ['nli_se_only', 'embedding_se_only', 'flare_only',
                     'flare_plus_nli_se', 'flare_plus_emb_se', 'all_features']
    print(f'\n## Cross-model AUC table (filtered subset, LOO-LR with bootstrap 95% CI)\n')
    headers = ['arm'] + [m.split('/')[-1] for m in all_results.keys()]
    print(' | '.join(headers))
    print(' | '.join(['---'] * len(headers)))
    for arm in HEADLINE_ARMS:
        row = [arm]
        for m, r in all_results.items():
            a = r['arm_auc'].get(arm, {})
            if not a or a.get('auc') is None:
                row.append('—')
                continue
            row.append(f"{a['auc']:.2f} [{a['ci_lo']:.2f},{a['ci_hi']:.2f}]")
        print(' | '.join(row))

    print(f'\n## Cross-model overall accuracy + filter retention\n')
    print('model | never | always | gap | filtered n | positive rate')
    print(' | '.join(['---'] * 6))
    for m, r in all_results.items():
        ov, fs = r['overall'], r['filtered_subset']
        print(f"{m.split('/')[-1]} | {ov['never_retrieve_accuracy']:.3f} | "
              f"{ov['always_retrieve_accuracy']:.3f} | {ov['gap_pp']:+.1f}pp | "
              f"{fs['n']} | {fs['positive_rate']:.1%}")

    # Save aggregated
    aggregated_path = f'{OUT_DIR}/v5_aggregated.json'
    with open(aggregated_path, 'w') as fh:
        json.dump(all_results, fh, indent=2)
    print(f'\nsaved aggregated cross-model results to {aggregated_path}')

## Phase 4 — Gated retrieval cost/accuracy curves

For each model and each gating signal, sweep thresholds and report (retrieval_rate, accuracy) pairs. This is the *systems-level* answer: how much accuracy do you trade for how much compute saved.

Run this cell after Phase 3 completes; it loads the saved per-model curves and prints summary points.

In [ ]:
import json
from pathlib import Path
import numpy as np

out_dir = Path(OUT_DIR)
results_files = sorted(out_dir.glob('results__*.json'))

def curve_at_retrieval_rate(curve, target_rate):
    """Return the curve point with retrieval_rate closest to target."""
    if not curve:
        return None
    rates = np.array([c['retrieval_rate'] for c in curve])
    idx = int(np.argmin(np.abs(rates - target_rate)))
    return curve[idx]

for f in results_files:
    with open(f) as fh:
        r = json.load(fh)
    print(f"\n=== {r['model']} ===")
    fs = r['filtered_subset']
    print(f"  (filtered subset n={fs['n']}, never-retrieve acc on subset = "
          f"{fs['never_retrieve_accuracy']:.3f}, always = 1.000 by filter)")
    headline_curves = ['arm__nli_se_only', 'arm__embedding_se_only',
                       'arm__flare_only', 'arm__flare_plus_nli_se',
                       'univariate__nli_se', 'univariate__semantic_entropy']
    print(f"\n  Operating points (retrieval rate → accuracy on filtered set):")
    print(f"    {'signal':<26} {'@25%':>8} {'@50%':>8} {'@75%':>8} {'@100%':>8}")
    for sig in headline_curves:
        curve = r['gated_retrieval_curves'].get(sig)
        if not curve:
            continue
        row = [sig.replace('arm__', '').replace('univariate__', 'u_')]
        for rate in [0.25, 0.50, 0.75, 1.0]:
            pt = curve_at_retrieval_rate(curve, rate)
            row.append(f"{pt['accuracy']:.3f}" if pt else '—')
        print(f"    {row[0]:<26} {row[1]:>8} {row[2]:>8} {row[3]:>8} {row[4]:>8}")

## What to do next

**Per-model results land at:** `/content/drive/MyDrive/cce_poc_v5/results__{slug}.json`. Download all of them to the repo at `research/paper/v5_results/` after the run.

**To check the run was clean:**
- Per-model overall `gap_pp` should be > 10pp (RepoBench is genuinely retrieval-needed).
- Filtered subset size should be ≥ 30 per model.
- AUCs that beat the v3 ceiling on this benchmark unlock the cross-domain story for the paper.

**If a model's filter retention is too low (e.g., < 10%):** that model can't solve RepoBench cross-file completion well even with retrieval. The signal-quality question becomes moot for that model on this benchmark; report and move on. CodeLlama is the most likely failure case here (smallest, weakest of the three).

**If all three models give clean results:** the paper now has a multi-model, contamination-resistant validation of the gating-signal claim. That's the Findings-tier result we set out to get.